In [28]:
# Kirchhoff-Love Platte (Hellan–Herrmann–Johnson, HHJ) in NGSolve
# Problem:  div div( D * hess(w) ) = q   (uniforme Eigenlast)
# Simply supported: w = 0  und  M_nn = 0  (=> sigma_nn = 0) am gesamten Rand

from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry

# -----------------------------
# Geometrie / Netz
# -----------------------------
a = 3.0      # Länge [m]
b = 2.0      # Breite [m]
radius = 0.06
maxh = 0.03  # Netzweite

# geo = SplineGeometry()
#geo.AddRectangle((0,0), (a,b), bc="ss")   # alle Kanten als "ss"

# #2seiten eingespannt: Randbedingungen Rechteck mit getrennten Rändern
# p1 = geo.AppendPoint(0, 0)
# p2 = geo.AppendPoint(a, 0)
# p3 = geo.AppendPoint(a, b)
# p4 = geo.AppendPoint(0, b)

# # Beispiel: x=0 und x=a => "ss" (einfach gelagert), y=0 und y=b => "free"
# geo.Append(["line", p1, p2], bc="free")  # y=0
# geo.Append(["line", p2, p3], bc="ss")    # x=a
# geo.Append(["line", p3, p4], bc="free")  # y=b
# geo.Append(["line", p4, p1], bc="ss")    # x=0


rect = MoveTo(0,0).Rectangle(a,b).Face()
support = [(a/5,4*b/5),(3*a/5,4*b/5),(4*a/5,b/5),(4*a/5,3*b/20)]

rect.edges.name="free"
#circ1 = Circle((0.5,0.5),0.2).Face()
for points in support:
    circ = Circle(points,radius).Face()
    rect = rect - circ
    circ.edges.name="ss"

geo = rect
#circ1.edges.name="ss"
#geo = rect - circ1
geo = OCCGeometry(geo,dim=2)


mesh = Mesh(geo.GenerateMesh(maxh=maxh))
mesh.Curve(3)
Draw(mesh)




WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [29]:
# -----------------------------
# Material / Last
# -----------------------------
E  = 70e9      # Glas ~ 70 GPa Elastizitätsmodul
nu = 0.35
t  = 0.02     # Dicke [m]
rho = 2699.89   # kg/m^3 Dichte einer Aluminiumplatte
g = 9.81

q = rho * g * t          # Eigengewicht als Flächenlast [N/m^2]

# Materialtensor (Inverse) für Kirchhoff-Love:
# D(eps) = Db * ((1-nu)*eps + nu*tr(eps)*I),  Db = E t^3 / (12(1-nu^2))
# Hier brauchen wir D^{-1} in der Bilinearform:
Db = E * t**3 / (12*(1-nu**2))


In [30]:
def DInv(mat):
    # D^{-1}(mat)
    return (1/Db) * ( (1/(1-nu))*mat - (nu/(1-nu**2))*Trace(mat)*Id(2) )

# -----------------------------
# HHJ-FEM: sigma ~ D*hess(w), w in H1
# Simply supported:
#   w = 0  => Q dirichlet="ss"
#   M_nn=0 => sigma_nn=0  => V dirichlet="ss"  (bei HDivDiv bedeutet das nn-Komponente)
# -----------------------------
order = 3
V = HDivDiv(mesh, order=order-1, dirichlet="ss")   # sigma_nn = 0 auf Rand
Q = H1(mesh, order=order,   dirichlet="ss")        # w = 0 auf Rand
X = V * Q

(sigma, w), (tau, v) = X.TnT()
n = specialcf.normal(2)

def tang(u):
    return u - (u*n)*n

aform = BilinearForm(X, symmetric=True)

# HHJ-Formulierung (wie in NGSolve-Doku):
aform += (
    InnerProduct(DInv(sigma), tau)
    + div(sigma) * Grad(v)
    + div(tau)   * Grad(w)
    ) * dx \
- (sigma[n,:] * tang(Grad(v)) + tau[n,:] * tang(Grad(w))) * dx(element_boundary=True)

aform.Assemble()

lform = LinearForm(X)
lform += q * v * dx
lform.Assemble()

gfu = GridFunction(X)
gfu.vec.data = aform.mat.Inverse(X.FreeDofs(), inverse="") * lform.vec

gf_sigma, gf_w = gfu.components

Draw(gf_w, mesh, name="w (deflection)")
Draw(gf_sigma, mesh, name="sigma (bending moment tensor)")


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene